## 🎯 Learning Objectives
* Understand the foundational architecture and key innovations of LeNet-5.
* Grasp the architectural components and groundbreaking contributions of AlexNet.
* Compare and contrast LeNet-5 and AlexNet, recognizing their evolutionary significance in deep learning.
* Implement simplified versions of LeNet-5 and AlexNet using PyTorch.


# Welcome to DL01-L15: Classic Architectures - LeNet & AlexNet Overview

Welcome, future Deep Learning architects! In this lesson, DL01-L15, we'll embark on a journey through the foundational convolutional neural network (CNN) architectures that paved the way for modern AI breakthroughs: LeNet-5 and AlexNet.

This course, **DL-01: Introduction to Deep Learning using PyTorch**, is designed for experienced ML engineers transitioning into the exciting world of deep learning. You're already proficient in traditional machine learning (ML-02) and Python, and now we're equipping you with the skills to build and train neural networks from scratch using PyTorch.

**Why this lesson matters:** Understanding these classic architectures isn't just about history; it's about grasping the fundamental principles and design patterns that underpin virtually all state-of-the-art CNNs today. LeNet-5 introduced the core concepts of CNNs, while AlexNet demonstrated their unprecedented power on large-scale image recognition tasks, igniting the deep learning revolution. By dissecting these models, you'll gain invaluable intuition for designing and optimizing your own deep learning solutions.

**How it fits:** We've already covered the basics of neural networks, backpropagation, and the core components of CNNs (convolutional layers, pooling, activation functions). This lesson builds directly on that knowledge, showing you how these components are assembled into powerful, end-to-end systems. It's a crucial stepping stone before we dive into even deeper and more complex architectures like VGG, ResNet, and Transformers.


## Prerequisites and Tools Check

Before we dive in, let's ensure our environment is ready. This lesson assumes you have:

*   **Python Proficiency:** Solid understanding of Python programming.
*   **ML-02 Knowledge:** Familiarity with traditional machine learning concepts.
*   **DL-01 Previous Lessons:** A good grasp of basic neural networks, backpropagation, and the fundamental building blocks of CNNs (convolutional layers, pooling, activation functions).

**Required Libraries (as of 2026):**

*   **PyTorch (torch):** The deep learning framework we'll be using.
*   **Torchvision:** Provides datasets, models, and image transformations.
*   **Matplotlib:** For basic plotting and visualization.
*   **NumPy:** For numerical operations.

**Environment Setup:**

We highly recommend running this notebook in a Google Colab environment or a local setup with GPU acceleration. Deep learning models, even simplified ones, benefit significantly from GPU computation. Colab typically provides free access to NVIDIA GPUs, making it an excellent choice for learning and experimentation.


In [ ]:
# Install necessary libraries if you're running this locally and haven't already
# In Google Colab, these are usually pre-installed or can be installed quickly.
# !pip install torch torchvision matplotlib numpy

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np

print(f"PyTorch Version: {torch.__version__}")

# Check for CUDA (GPU) availability
if torch.cuda.is_available():
    print(f"CUDA is available! Using GPU: {torch.cuda.get_device_name(0)}")
    device = torch.device("cuda")
else:
    print("CUDA is not available. Using CPU.")
    device = torch.device("cpu")

# A simple 'hello world' for PyTorch: create a tensor and move it to the device
x = torch.randn(3, 3).to(device)
print(f"\nHello PyTorch! A random tensor on {device}:\n{x}")

# Verify basic CNN building blocks
conv_layer = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, stride=1, padding=1).to(device)
input_tensor = torch.randn(1, 3, 32, 32).to(device) # Batch, Channels, Height, Width
output_tensor = conv_layer(input_tensor)
print(f"\nBasic Conv2d layer output shape: {output_tensor.shape}")

print("Environment setup complete. Ready to explore classic CNN architectures!")


## The Dawn of CNNs: LeNet-5

Our journey begins with **LeNet-5**, a pioneering convolutional neural network architecture developed by Yann LeCun and his team in 1998. While deep learning gained widespread attention much later, LeNet-5 was a groundbreaking achievement, demonstrating the power of CNNs for image recognition, specifically for handwritten digit recognition (e.g., ZIP codes, bank checks).

**Key Innovations and Characteristics of LeNet-5:**

1.  **Convolutional Layers:** Introduced the concept of local receptive fields and shared weights, allowing the network to learn hierarchical features from images efficiently.
2.  **Subsampling (Pooling) Layers:** Used average pooling to reduce spatial dimensions, making the network more robust to small translations and distortions.
3.  **Activation Functions:** Employed `tanh` (hyperbolic tangent) activation functions, which were common at the time.
4.  **Architecture:** A relatively shallow network consisting of alternating convolutional and subsampling layers, followed by fully connected layers.
5.  **Application:** Primarily designed for MNIST digit classification, achieving remarkable accuracy for its time.

LeNet-5 laid the fundamental blueprint for almost all subsequent CNN architectures. Its principles of feature extraction through convolution and spatial reduction through pooling remain central to modern deep learning.


In [ ]:
print("\n--- LeNet-5 Architecture Sketch ---")

class LeNet5(nn.Module):
    def __init__(self):
        super(LeNet5, self).__init__()
        # LeNet-5 expects 32x32 grayscale input, but we'll adapt for common 1-channel input
        # C1: Convolutional Layer (6 feature maps, 5x5 kernel)
        # Input: 1x32x32 (MNIST-like) -> Output: 6x28x28
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=6, kernel_size=5, stride=1, padding=0)
        # S2: Average Pooling Layer (2x2 kernel, stride 2)
        # Input: 6x28x28 -> Output: 6x14x14
        self.pool1 = nn.AvgPool2d(kernel_size=2, stride=2)
        
        # C3: Convolutional Layer (16 feature maps, 5x5 kernel)
        # Note: LeNet-5 had a more complex connection table here, but we'll use full connectivity for simplicity
        # Input: 6x14x14 -> Output: 16x10x10
        self.conv2 = nn.Conv2d(in_channels=6, out_channels=16, kernel_size=5, stride=1, padding=0)
        # S4: Average Pooling Layer (2x2 kernel, stride 2)
        # Input: 16x10x10 -> Output: 16x5x5
        self.pool2 = nn.AvgPool2d(kernel_size=2, stride=2)
        
        # F5: Fully Connected Layer (120 units)
        # Input: 16*5*5 = 400 -> Output: 120
        self.fc1 = nn.Linear(in_features=16 * 5 * 5, out_features=120)
        # F6: Fully Connected Layer (84 units)
        # Input: 120 -> Output: 84
        self.fc2 = nn.Linear(in_features=120, out_features=84)
        # Output Layer (10 units for digits 0-9)
        # Input: 84 -> Output: 10
        self.fc3 = nn.Linear(in_features=84, out_features=10)
        
        # Activation functions (LeNet used tanh, we'll use it here for historical accuracy)
        self.tanh = nn.Tanh()

    def forward(self, x):
        # C1 -> Tanh -> S2
        x = self.tanh(self.conv1(x))
        x = self.pool1(x)
        
        # C3 -> Tanh -> S4
        x = self.tanh(self.conv2(x))
        x = self.pool2(x)
        
        # Flatten for fully connected layers
        x = x.view(-1, 16 * 5 * 5) # -1 infers batch size
        
        # F5 -> Tanh
        x = self.tanh(self.fc1(x))
        # F6 -> Tanh
        x = self.tanh(self.fc2(x))
        
        # Output layer (no activation here, typically softmax applied later for classification)
        x = self.fc3(x)
        return x

# Instantiate the model and move to device
lenet_model = LeNet5().to(device)
print(lenet_model)

# Test with a dummy input (batch size 1, 1 channel, 32x32 image)
dummy_input_lenet = torch.randn(1, 1, 32, 32).to(device)
output_lenet = lenet_model(dummy_input_lenet)

print(f"\nInput shape: {dummy_input_lenet.shape}")
print(f"Output shape (logits for 10 classes): {output_lenet.shape}")
print("LeNet-5 forward pass successful!")


## The Deep Learning Revolution: AlexNet

Fast forward to 2012, and the landscape of computer vision was dramatically reshaped by **AlexNet**. Developed by Alex Krizhevsky, Ilya Sutskever, and Geoffrey Hinton, AlexNet won the ImageNet Large Scale Visual Recognition Challenge (ILSVRC) by a significant margin, achieving a top-5 error rate of 15.3% compared to the second-place entry's 26.2%. This victory was a watershed moment, demonstrating the unprecedented power of deep convolutional neural networks on large, complex datasets.

**Key Innovations and Characteristics of AlexNet:**

1.  **Depth and Width:** Significantly deeper and wider than LeNet-5, with 8 layers (5 convolutional, 3 fully connected).
2.  **ReLU Activation:** Replaced `tanh` with the Rectified Linear Unit (ReLU) activation function. ReLU addressed the vanishing gradient problem, allowing deeper networks to be trained more effectively and speeding up training significantly.
3.  **Dropout:** Introduced Dropout regularization to prevent overfitting, especially crucial for a model with millions of parameters.
4.  **Data Augmentation:** Extensively used data augmentation techniques (random cropping, horizontal flipping, color jittering) to artificially expand the training dataset and improve generalization.
5.  **GPU Utilization:** Designed to be trained across two GPUs, a necessity for its size and the computational demands of ImageNet. This highlighted the importance of parallel computing in deep learning.
6.  **Overlapping Max Pooling:** Used max pooling with a stride smaller than the kernel size, leading to overlapping pooling regions, which was found to improve accuracy.
7.  **Local Response Normalization (LRN):** An early form of normalization (though largely superseded by Batch Normalization later) that helped with generalization.

AlexNet's success proved that deep learning was not just a theoretical concept but a practical solution for real-world, large-scale image recognition problems, sparking the modern AI boom.


In [ ]:
print("\n--- AlexNet Architecture Sketch ---")

class AlexNet(nn.Module):
    def __init__(self, num_classes=1000):
        super(AlexNet, self).__init__()
        # AlexNet was designed for 224x224 or 227x227 RGB images
        self.features = nn.Sequential(
            # 1st Conv Layer: 96 filters, 11x11 kernel, stride 4, padding 2
            # Input: 3x227x227 -> Output: 96x55x55
            nn.Conv2d(3, 96, kernel_size=11, stride=4, padding=2),
            nn.ReLU(inplace=True),
            # Max Pooling: 3x3 kernel, stride 2
            # Input: 96x55x55 -> Output: 96x27x27
            nn.MaxPool2d(kernel_size=3, stride=2),
            
            # 2nd Conv Layer: 256 filters, 5x5 kernel, stride 1, padding 2
            # Input: 96x27x27 -> Output: 256x27x27
            nn.Conv2d(96, 256, kernel_size=5, stride=1, padding=2),
            nn.ReLU(inplace=True),
            # Max Pooling: 3x3 kernel, stride 2
            # Input: 256x27x27 -> Output: 256x13x13
            nn.MaxPool2d(kernel_size=3, stride=2),
            
            # 3rd Conv Layer: 384 filters, 3x3 kernel, stride 1, padding 1
            # Input: 256x13x13 -> Output: 384x13x13
            nn.Conv2d(256, 384, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            
            # 4th Conv Layer: 384 filters, 3x3 kernel, stride 1, padding 1
            # Input: 384x13x13 -> Output: 384x13x13
            nn.Conv2d(384, 384, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            
            # 5th Conv Layer: 256 filters, 3x3 kernel, stride 1, padding 1
            # Input: 384x13x13 -> Output: 256x13x13
            nn.Conv2d(384, 256, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            # Max Pooling: 3x3 kernel, stride 2
            # Input: 256x13x13 -> Output: 256x6x6
            nn.MaxPool2d(kernel_size=3, stride=2),
        )
        
        self.avgpool = nn.AdaptiveAvgPool2d((6, 6)) # Adapts any input size to 6x6
        
        self.classifier = nn.Sequential(
            nn.Dropout(p=0.5), # Dropout for regularization
            nn.Linear(256 * 6 * 6, 4096),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.5),
            nn.Linear(4096, 4096),
            nn.ReLU(inplace=True),
            nn.Linear(4096, num_classes) # Output for 1000 ImageNet classes
        )

    def forward(self, x):
        x = self.features(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1) # Flatten all dimensions except batch
        x = self.classifier(x)
        return x

# Instantiate the model and move to device
alexnet_model = AlexNet(num_classes=1000).to(device)
print(alexnet_model)

# Test with a dummy input (batch size 1, 3 channels, 227x227 image)
dummy_input_alexnet = torch.randn(1, 3, 227, 227).to(device)
output_alexnet = alexnet_model(dummy_input_alexnet)

print(f"\nInput shape: {dummy_input_alexnet.shape}")
print(f"Output shape (logits for 1000 classes): {output_alexnet.shape}")
print("AlexNet forward pass successful!")


## Comparing LeNet-5 and AlexNet: An Evolutionary Leap

While both LeNet-5 and AlexNet are foundational CNNs, AlexNet represented a significant evolutionary leap, primarily driven by increased computational power and the availability of large datasets like ImageNet. Here's a comparison:

| Feature             | LeNet-5                                     | AlexNet                                         |
| :------------------ | :------------------------------------------ | :---------------------------------------------- |
| **Year**            | 1998                                        | 2012                                            |
| **Primary Task**    | Handwritten Digit Recognition (MNIST)       | Large-Scale Image Classification (ImageNet)     |
| **Input Size**      | 32x32 (grayscale)                           | 227x227 or 224x224 (RGB)                        |
| **Depth**           | 7 layers (2 conv, 2 pool, 3 FC)             | 8 layers (5 conv, 3 FC)                         |
| **Activation**      | `tanh`                                      | `ReLU` (Rectified Linear Unit)                  |
| **Pooling**         | Average Pooling (non-overlapping)           | Max Pooling (overlapping)                       |
| **Regularization**  | None explicitly                             | Dropout, Data Augmentation                      |
| **Computational**   | CPU-based                                   | GPU-accelerated (designed for 2 GPUs)           |
| **Parameters**      | ~60,000                                     | ~60 million                                     |
| **Impact**          | Pioneered CNNs, demonstrated feasibility    | Ignited the deep learning revolution, set new SOTA |

**Key Takeaways from the Evolution:**

*   **Deeper and Wider Networks:** The ability to train deeper networks was crucial for learning more complex, hierarchical features.
*   **ReLU's Importance:** ReLU was a game-changer, enabling faster training and mitigating vanishing gradients.
*   **Regularization is Key:** As models grew larger, techniques like Dropout became essential to prevent overfitting.
*   **Data is Fuel:** Large, diverse datasets (like ImageNet) combined with data augmentation were critical for training robust models.
*   **Hardware Matters:** The advent of powerful GPUs made training these massive models feasible, directly contributing to the deep learning boom.

These lessons learned from AlexNet continue to influence CNN design to this day.


## Conclusion and Next Steps

Congratulations! You've successfully navigated the foundational architectures of convolutional neural networks. We've explored:

*   **LeNet-5:** The pioneering CNN that introduced core concepts like convolutional layers, shared weights, and subsampling, proving the viability of neural networks for image recognition.
*   **AlexNet:** The revolutionary architecture that, powered by GPUs and large datasets, demonstrated the unprecedented capabilities of deep CNNs, kickstarting the modern deep learning era with innovations like ReLU, Dropout, and extensive data augmentation.

Understanding these classic models provides a crucial historical context and practical intuition for the design principles that govern modern deep learning. You've seen how the basic building blocks of CNNs are assembled into powerful systems and how architectural innovations drive progress.

**What's next?**

In the upcoming lessons, we'll continue our exploration of CNN architectures, delving into models that built upon AlexNet's success, such as:

*   **VGGNet:** Known for its simplicity and uniform architecture with very small convolutional filters.
*   **ResNet (Residual Networks):** A breakthrough that enabled the training of extremely deep networks through residual connections.
*   **Inception (GoogLeNet):** Introduced the concept of 'inception modules' for efficient computation and multi-scale feature extraction.

These architectures will further deepen your understanding of how to design efficient, powerful, and robust deep learning models for a wide range of computer vision tasks. Keep up the great work!
